In [ ]:
# Step 1: Upload your kaggle.json API token
from google.colab import files
files.upload()

# Step 2: Move kaggle.json to correct location and set permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Step 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 4: Download dataset using Kaggle API
# Replace 'username/dataset-name' with the actual Kaggle dataset path
!kaggle datasets download -d amananandrai/ag-news-classification-dataset --unzip

# Step 5: Move downloaded dataset to Google Drive (MyDrive)
# Replace 'dataset-folder-or-files' with actual dataset folder name or filenames
# Assuming the dataset is unzipped into a folder named 'ag-news-classification-dataset'
!mv ag-news-classification-dataset /content/drive/MyDrive/Deployment_of_News_Article/

Saving kaggle.json to kaggle (3).json
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset URL: https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset
License(s): unknown
  0% 0.00/11.4M [00:00<?, ?B/s]
100% 11.4M/11.4M [00:00<00:00, 934MB/s]


##**MODEL TUNING**

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    TrainerCallback
)

In [ ]:
# --- 2. Login to W&B ---
import wandb


In [ ]:
# W&B Config
ENTITY = "sinwanmohammed022-guvi-geek-networks"
PROJECT_NAME = "ag-news-classification"
DATASET_ARTIFACT_NAME = "ag-news-dataset"
MODEL_ARTIFACT_NAME = "ag-news-distilbert"

In [ ]:
run = wandb.init(entity=ENTITY,project=PROJECT_NAME, job_type="model-training",resume = "allow")

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sinwanmohammed022 (sinwanmohammed022-guvi-geek-networks) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
#Load Dataset from Google Drive ------------------
dataset_dir = "/content/drive/MyDrive/Deployment_of_News_Article"
train_path = os.path.join(dataset_dir, "train.csv")
test_path = os.path.join(dataset_dir, "test.csv")

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

In [ ]:
train_df.head(5)


,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [ ]:
test_df.head(5)

,Class Index,Title,Description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...


In [ ]:

# Rename columns for consistency
train_df.columns = ["label", "title", "description"]
test_df.columns = ["label", "title", "description"]

In [ ]:
import re
import string

# Function to remove punctuation
def clean_text(text):
    return re.sub(f"[{re.escape(string.punctuation)}]", "", text).lower()

# Apply cleaning to title and description columns
for col in ["title", "description"]:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)


In [ ]:
def remove_whitespace(text):
    if isinstance(text, str):
        text = text.strip()                    # remove leading/trailing spaces
        text = " ".join(text.split())          # remove extra spaces, keep single spaces
    return text

# Apply to your dataframe
train_df["title"] = train_df["title"].apply(remove_whitespace)
train_df["description"] = train_df["description"].apply(remove_whitespace)

test_df["title"] = test_df["title"].apply(remove_whitespace)
test_df["description"] = test_df["description"].apply(remove_whitespace)

In [ ]:
train_df.head(5)

,label,title,description
0,3,wall st bears claw back into the black reuters,reuters shortsellers wall streets dwindlingban...
1,3,carlyle looks toward commercial aerospace reuters,reuters private investment firm carlyle groupw...
2,3,oil and economy cloud stocks outlook reuters,reuters soaring crude prices plus worriesabout...
3,3,iraq halts oil exports from main southern pipe...,reuters authorities have halted oil exportflow...
4,3,oil prices soar to alltime record posing new m...,afp tearaway world oil prices toppling records...


In [ ]:
test_df.head(5)

,label,title,description
0,3,fears for t n pension after talks,unions representing workers at turner newall s...
1,4,the race is on second private team sets launch...,spacecom toronto canada a secondteam of rocket...
2,4,ky company wins grant to study peptides ap,ap a company founded by a chemistry researcher...
3,4,prediction unit helps forecast wildfires ap,ap its barely dawn when mike fitzpatrick start...
4,4,calif aims to limit farmrelated smog ap,ap southern californias smogfighting agency we...


In [ ]:
# Fix labels: shift from 1-4 → 0-3
train_df["label"] = train_df["label"].astype(int) - 1
test_df["label"] = test_df["label"].astype(int) - 1

In [ ]:
# Merge title + description into a single text field
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"] = test_df["title"] + " " + test_df["description"]

In [ ]:
# Log Dataset to W&B (only once needed) ------------------
dataset_artifact = wandb.Artifact(DATASET_ARTIFACT_NAME, type="dataset")
dataset_artifact.add_file(train_path)
dataset_artifact.add_file(test_path)
run.log_artifact(dataset_artifact)

<Artifact ag-news-dataset>

In [ ]:
# Convert to HF Dataset ------------------
train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

In [ ]:
model_name = "distilbert-base-uncased"

In [ ]:
#Tokenization ------------------

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
# Model ------------------
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:


def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

In [ ]:
from transformers import TrainerCallback
import glob

class WandbCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        # Find latest checkpoint folder
        checkpoints = sorted(glob.glob(os.path.join(args.output_dir, "checkpoint-*")), key=os.path.getmtime)
        if checkpoints:
            latest_ckpt = checkpoints[-1]
            artifact = wandb.Artifact(MODEL_ARTIFACT_NAME, type="model-checkpoint")
            artifact.add_dir(latest_ckpt)
            wandb.log_artifact(artifact)
            print(f"✅ Logged checkpoint to W&B: {latest_ckpt}")


In [ ]:
#  Training Arguments ------------------
output_dir = "/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints"

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=500,
    save_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="wandb"
)

In [ ]:
# Trainer ------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    callbacks=[WandbCheckpointCallback()]
)


/tmp/ipython-input-4186619766.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Auto-Detect Resume Checkpoint ------------------
def get_latest_checkpoint(entity, project, model_artifact_name):
    api = wandb.Api()

    try:
        artifact_type = api.artifact_type(
            type_name="model-checkpoint",
            project=f"{entity}/{project}"
        )
        collection = artifact_type.collection(model_artifact_name)
        versions = sorted(
            collection.versions(),
            key=lambda v: int(v.name.split("-")[-1]),
            reverse=True
        )
        if not versions:
            print("No checkpoints found on W&B — starting fresh.")
            return None
        latest_ckpt = versions[0]
        print(f"Resuming from latest W&B checkpoint: {latest_ckpt.name}")
        return latest_ckpt.download()
    except Exception as e:
        print(f"Auto-resume lookup failed: {e}")
        return None

In [ ]:
import wandb
run = wandb.init()
artifact = run.use_artifact('sinwanmohammed022-guvi-geek-networks/ag-news-classification/ag-news-distilbert:v2', type='model-checkpoint')
artifact_dir = artifact.download()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sinwanmohammed022 (sinwanmohammed022-guvi-geek-networks) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Downloading large artifact ag-news-distilbert:v2, 767.27MB. 11 files... 
wandb:   11 of 11 files downloaded.  
Done. 0:0:18.5 (41.5MB/s)


In [ ]:
artifact_dir = "/content/artifacts/ag-news-distilbert:v2"

In [ ]:
trainer.train(resume_from_checkpoint=artifact_dir)


Step,Training Loss,Validation Loss,Accuracy
2000,0.255100,0.270062,0.918026
2500,0.259400,0.254379,0.923421


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-2000)... Done. 13.6s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-2000


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-2500)... Done. 11.9s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-2500


In [ ]:
import wandb
run = wandb.init()
artifact = run.use_artifact('sinwanmohammed022-guvi-geek-networks/ag-news-classification/run-3pgyhmj4-history:v0', type='wandb-history')
artifact_dir = artifact.download()

wandb:   1 of 1 files downloaded.  


In [ ]:
checkpoint_dir = "/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-3000"

In [ ]:
trainer.train(resume_from_checkpoint=checkpoint_dir)

Step,Training Loss,Validation Loss,Accuracy
3500,0.234200,0.234150,0.922237
4000,0.239100,0.226663,0.927500
4500,0.236200,0.220712,0.928947
5000,0.213500,0.227005,0.928026


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-3500)... Done. 25.7s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-3500


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-4000)... Done. 20.2s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-4000


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-4500)... Done. 31.6s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-4500


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-5000)... Done. 26.1s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-5000


In [ ]:
checkpoint_dir = "/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-5500"

In [ ]:
trainer.train(resume_from_checkpoint=checkpoint_dir)

Step,Training Loss,Validation Loss,Accuracy
6000,0.212700,0.222671,0.931974
6500,0.218800,0.210384,0.931184
7000,0.203900,0.213000,0.934605


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-6000)... Done. 21.3s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-6000


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-6500)... Done. 17.8s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-6500


...

Step,Training Loss,Validation Loss,Accuracy
6000,0.212700,0.222671,0.931974
6500,0.218800,0.210384,0.931184
7000,0.203900,0.213000,0.934605
7500,0.214000,0.200507,0.935263
8000,0.163500,0.215103,0.934737


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-7500)... Done. 20.8s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-7500


wandb: Adding directory to artifact (/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-8000)... Done. 24.3s


✅ Logged checkpoint to W&B: /content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-8000


In [ ]:
checkpoint_path = "/content/drive/MyDrive/Deployment_of_News_Article/ag_news_checkpoints/checkpoint-7500"

In [ ]:
# ------------------ 15. Save Final Model ------------------
from transformers import AutoModelForSequenceClassification, AutoTokenizer
# Load model & tokenizer from checkpoint
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)


In [ ]:
final_model_path = "/content/drive/MyDrive/Deployment_of_News_Article/final_model"
# Save as final model
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

('/content/drive/MyDrive/Deployment_of_News_Article/final_model/tokenizer_config.json',
 '/content/drive/MyDrive/Deployment_of_News_Article/final_model/special_tokens_map.json',
 '/content/drive/MyDrive/Deployment_of_News_Article/final_model/vocab.txt',
 '/content/drive/MyDrive/Deployment_of_News_Article/final_model/added_tokens.json',
 '/content/drive/MyDrive/Deployment_of_News_Article/final_model/tokenizer.json')